<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Manipulation de données satellitaires - Format NetCDF</div>

---

## 🎯 **Objectifs du TP**


## 🔎  Analyser un fichier NetCDF 
## 🖼️  Produire une image à partir d'un canal 
## 🎚️  Modifier la dynamique des valeurs 
## 📍  Extraire une valeur ponctuelle 

---

## 📑 **Table des matières**

| Section | Titre |
|---------|-------|
| 🔗 | [**1.** 🔎 Visualisation du contenu du NetCDF](#1-visualisation-du-contenu) |
| 🔗 | [**2.** 🖼️ Production d'une image TIF (canal VIS006)](#2-production-dune-image-tif-canal-vis006) |
| 🔗 | [**3.** 🌡️ Production d'une image TIF (canal IR_105)](#3-production-dune-image-tif-canal-ir_105) |
| 🔗 | [**4.** 📍 Extraction de valeurs ponctuelles](#4-extraction-de-valeurs-ponctuelles) | 
| 🔗 | [**5.** 📍 Extraction de valeurs ponctuelles à partir des TIF](#5-extraction-de-valeurs-ponctuelles) |
| 🔗 | [**6.** 📍 Exemples réalisés à partir d'extractions](6-exemples) |

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise en grande partie la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **NetCDF**, issus de la production opérationnelle de Météo-France.

---

## 📂 **Fichiers utilisés**

| Fichier | Résolution | Description |
|---------|------------|-------------|
| `Mmultic500mNC4_mtgi1_202604221200.nc` | 500 m | 2 canaux |
| `Mmultic1kmNC4_mtgi1_202604221200.nc` | 1 km | 10 canaux |
| `Mmultic2kmNC4_mtgi1_202604221200.nc` | 2 km | 16 canaux |

📁 **Répertoire** : `/stockage/DATA/NetCDF/20260422`

> 💡 *Ces fichiers multicanaux sont produits par le Centre de Météorologie Spatiale (CMS) de Météo-France pour les satellites géostationnaires.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

## 📡 **Rappel des différents canaux**

<div align="center">
<a href="../DOCS/Canaux.png" target="_blank">
    <img src='../DOCS/Canaux.png' alt='Canaux MTG' width='900px'>
</a>
</div>

In [ ]:
Mmultic500m="/stockage/DATA/NetCDF/20260422/Mmultic500mNC4_mtgi1_202604221200.nc"
Mmultic1km="/stockage/DATA/NetCDF/20260422/Mmultic1kmNC4_mtgi1_202604221200.nc"
Mmultic2km="/stockage/DATA/NetCDF/20260422/Mmultic2kmNC4_mtgi1_202604221200.nc"

print("📁 Fichiers configurés :")
print(f"   - 500m : {Mmultic500m}")
print(f"   - 1km  : {Mmultic1km}")
print(f"   - 2km  : {Mmultic2km}")

<div id="1-visualisation-du-contenu"></div>
<div class="alert alert-info alert-success">
<h3 id="1"> 1️⃣ - 🔎 Visualisation du contenu du NetCDF</h3>
</div>

La commande `gdalinfo` permet d'afficher les métadonnées d'un fichier NetCDF. Dans Jupyter, on la précède de `!`.

In [ ]:
!gdalinfo $Mmultic2km 2>/dev/null

## 📋 **Décryptage des informations**

### 🏷️ **Informations générales**

| Élément | Valeur | Signification |
|---------|--------|----------------|
| 🏷️ **Driver** | netCDF | Format scientifique multidimensionnel |
| 📛 **Source** | MTG (Meteosat Third Generation) | Satellite Meteosat-12 |
| 🖼️ **Couverture** | globe | Disque terrestre complet |
| 🕒 **Date création** | 2026-04-22 12:05 | Production post-acquisition |
| 🕒 **Acquisition** | 12:00:07 → 12:09:34 | Scan de ~9 minutes |
| 🏷️ **Institution** | Météo-France (CMS Lannion) | Producteur |

### 📦 **Structure du fichier**

- Le fichier contient plusieurs **SUBDATASETS** (un par canal)
- Chaque SUBDATASET est une image de **5568 × 5568 pixels**
- Résolutions disponibles : 500 m, 1 km, 2 km
- Géolocalisation (approx) lat: -80 → +80 lon: -80 → +80 → Domaine visible depuis satellite géostationnaire

### 🎨 **Canaux disponibles (2 km)**

| Type | Canaux | Grandeur |
|------|--------|----------|
| **Visible** | VIS004, VIS005, VIS006, VIS008, VIS009 | Réflectance TOA |
| **Proche IR** | IR_013, IR_016, IR_022 | Réflectance ou température |
| **Infrarouge** | IR_038, WV_063, WV_073, IR_087, IR_097, IR_105, IR_123, IR_133 | Température de brillance |
| **Temporel** | dtime | Temps relatif par pixel (s)|

In [ ]:
!gdalinfo $Mmultic1km 2>/dev/null | grep -E "SUBDATASET|Size"

> ℹ️ **Résolutions** :
> - **2 km** : 16 canaux
> - **1 km** : 10 canaux
> - **500 m** : 2 canaux

In [ ]:
!gdalinfo NETCDF:"$Mmultic2km":VIS006 2>/dev/null

## 🛰️ **Détail de la projection GEOS**

| Paramètre | Valeur | Explication |
|-----------|--------|-------------|
| **Projection** | Geostationary Satellite (Sweep Y) | Vue depuis l'espace |
| **Longitude origine** | 0° | Satellite au-dessus de Greenwich |
| **Altitude** | 35 786 400 m | Orbite géostationnaire |
| **Origine image** | (-5 568 000, 5 568 000) m | Coin haut gauche |
| **Pixel Size** | (2 000, -2 000) m | Résolution 2 km |
| **Centre** | (0, 0) → (0°E, 0°N) | Point sous-satellite |

> ⚠️ **Attention** : Les axes ne sont PAS en latitude/longitude ! Ce sont des coordonnées projetées en mètres.

## 🌡️ **Calibration du canal VIS006**

| Propriété | Valeur |
|-----------|--------|
| **Longueur d'onde** | 0.640 µm (rouge visible) |
| **Grandeur** | toa_bidirectional_reflectance |
| **scale_factor** | 0.01 |
| **add_offset** | 0 |
| **Formule** | Réflectance (%) = DN × 0.01 (stockée en int16)|
| **_FillValue** | -32768 (pixels sans donnée valide, hors disque) |

> 💡 *Exemple : DN = 10 000 → Réflectance = 100,0%*

In [ ]:
!gdalinfo -mm NETCDF:"$Mmultic2km":VIS006 2>/dev/null | grep -E "Size|Computed Min/Max|Pixel Size|Origin"

> 💡 *L'option `-mm` affiche les valeurs minimales et maximales trouvées dans le dataset.*

**Interprétation des valeurs** :
- `Computed Min/Max = -9707.000, 5678.000` (pour IR_105)
- `Computed Min/Max = 264.000, 12558.000` (pour VIS006)

⚠️ *Les réflectivités supérieures à 10 000 (>100 %) peuvent concerner des nuages glacés épais avec un soleil rasant.*

Rmq: la commande `ncdump`permet égalemetn de lire les métadonées

In [ ]:
!ncdump -h /stockage/DATA/NetCDF/20260422/Mmultic2kmNC4_mtgi1_202604221200.nc 2>/dev/null | grep -A8 "VIS006"

<div id="2-production-dune-image-tif-canal-vis006"></div>
<div class="alert alert-info alert-success">
<h3 id="2"> 2️⃣ - 🖼️ Production d'une image TIF à partir du canal VIS006</h3>
</div>

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'
nom_fic = 'VIS006_mtg_20260422_1200'
redim = '500x500'

print(f"📁 Dossier de sortie : {output}")
print(f"📏 Redimensionnement : {redim}")

### 🎨 **Étirement linéaire (contrast stretching, ou linear scaling)**

Nous allons transformer les valeurs physiques (16 bits) en nuances de gris (256 valeurs - 8 bits) avec `gdal_translate` et l'option `-scale` :

```
-scale entrée_min entrée_max sortie_min sortie_max
```

In [ ]:
!gdal_translate -scale 264 12558 0 255 -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD.jpg 2>/dev/null
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD.jpg $output/VIS006_mtg_20260422_1200_FD_{redim}.jpg 2>/dev/null
im1 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_{redim}.jpg", 'r')
display(im1)

### 🎚️ **Ajustement de la dynamique (windowing)**

En modifiant les bornes d'entrée, on peut rendre l'image plus lumineuse.

In [ ]:
valeur_min_vis=264
valeur_max_vis=8000

!gdal_translate -scale {valeur_min_vis} {valeur_max_vis} 0 255 -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee.jpg 2>/dev/null
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee.jpg $output/VIS006_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg 2>/dev/null
im2 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg", 'r')
display(im2)

> ⚠️ **Limite** : Les valeurs de réflectance > 80 % (8000) sont saturées à 255 (blanc pur). On perd l'information dans les nuages épais.</br>
> Le blanc d'une grande partie des CB ou nuages élevés épais sera ainsi complètement saturé. Le signal sera perdu

### 📐 **Correction Gamma (éclaircissement non linéaire)**

La correction Gamma permet d'éclaircir les zones sombres sans saturer les zones claires.</br>
Cet ajustement permet de modifier la luminosité d’une image de façon non linéaire.</br>
On parle aussi de « facteur de contraste »

| Gamma | Effet |
|-------|-------|
| < 1 | Assombrit l'image |
| = 1 | Aucun changement |
| > 1 | Éclaircit l'image |

> 💡 *GDAL utilise l'option `-exponent` avec comme valeur `1/gamma`.*

In [ ]:
Gamma=2
ValGamma=1/Gamma

!gdal_translate -scale {valeur_min_vis} {valeur_max_vis} 0 255 -exponent {ValGamma} -ot byte NETCDF:"$Mmultic2km":VIS006 $output/VIS006_mtg_20260422_1200_FD_dyn_gamma_modifiee.jpg 2>/dev/null
!convert -resize {redim} $output/VIS006_mtg_20260422_1200_FD_dyn_gamma_modifiee.jpg $output/VIS006_mtg_20260422_1200_FD_dyn_gamma{Gamma}_{redim}.jpg 2>/dev/null
im3 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_dyn_gamma{Gamma}_{redim}.jpg", 'r')
display(im3)

### 🏭 **Comparaison avec le produit opérationnel** (Gamma=1,4)

In [ ]:
!convert -resize {redim} /stockage/DATA/TIF/20260422/globeMvis006GTP.mtg.20260422.1200.24.tif.LT[3] $output/VIS006_mtg_20260422_1200_oper.jpg 2>/dev/null
im4 = Image.open(f"{output}/VIS006_mtg_20260422_1200_oper.jpg", 'r')
display(im4)

> 💡 *Le produit opérationnel bénéficie de corrections supplémentaires : diffusion, angle solaire et parallaxe.*

### 🔍 **Zoom sur la France (comparaison)**

🇫🇷 Pour mieux voir les différences, faisons un zoom sur la France en projection orthographique.

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 /stockage/DATA/TIF/20260422/globeMvis006GTP.mtg.20260422.1200.24.tif.LT $output/VIS006_mtg_20260422_1200_oper_france.jpg 2>/dev/null
!gdal_translate -scale 264 12558 0 255 -ot byte NETCDF:"$Mmultic1km":VIS006 $output/VIS006_mtg_20260422_1200_FD_500m.tif 2>/dev/null
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 $output/VIS006_mtg_20260422_1200_FD_500m.tif $output/VIS006_mtg_20260422_1200_FD_500m_france.jpg 2>/dev/null

In [ ]:
im5 = Image.open(f"{output}/VIS006_mtg_20260422_1200_oper_france.jpg", 'r')
print("🏭 Version opérationnelle")
display(im5)
im6 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_500m_france.jpg", 'r')
print("🔧 Version non corrigée")
display(im6)

Comparaison avec Synopsis :

In [ ]:
im_synopsis= Image.open(f"./DOCS/VIS006_mtg_20260422_1200_oper_france_synopsis.jpg", 'r')
display(im_synopsis)

💡 Pour comparer les images, les ouvrir dans des nouveaux onglets, ou utliser cette page d'aide à la comparaison de la <a href="http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html" target="_blank"> GALERIE CMS</a> (Télécharger les 2 images et glisser/déposer)

💡 Synopsis applique une **palette de couleur** non linéaire de **128 teintes de gris**. Les teintes sombres sont éclaircies.

<div id="3-production-dune-image-tif-canal-ir_105"></div>
<div class="alert alert-info alert-success">
<h3 id="3"> 3️⃣ - 🌡️ Production d'une image TIF à partir du canal IR_105</h3>
</div>

### 📊 **Spécificités du canal IR_105**

| Propriété | Valeur |
|-----------|--------|
| **Longueur d'onde** | 10.5 µm (infrarouge thermique) |
| **Grandeur** | toa_brightness_temperature |
| **Unités** | Kelvin (K) |
| **scale_factor** | 0.01 |
| **add_offset** | 273.15 |
| **Formule** | Température (K) = DN × 0.01 + 273.15 |

> 💡 *Les valeurs sont stockées en 16 bits en centièmes de degré Celsius : -50°C = -5000*

In [ ]:
!gdalinfo -mm NETCDF:"$Mmultic2km":IR_105 2>/dev/null | grep -E "Computed Min/Max|Pixel Size"

In [ ]:
valeur_min_ir=-9707  # Valeur brute min -9707
valeur_max_ir=5678   # Valeur brute max 5678

!gdal_translate -scale {valeur_min_ir} {valeur_max_ir} 255 0 -ot byte NETCDF:"$Mmultic2km":IR_105 $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee.jpg 2>/dev/null
!convert -resize {redim} $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee.jpg $output/IR_105_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg 2>/dev/null
im7 = Image.open(f"{output}/IR_105_mtg_20260422_1200_FD_dyn_modifiee_{redim}.jpg", 'r')
display(im7)

### 🌍 **Convention de représentation des températures**

| Région | Convention | Dynamique (`-scale`) |
|--------|------------|----------------------|
| **Europe** | Nuages froids = blancs | `min max 255 0` |
| **USA** | Nuages chauds = blancs | `min max 0 255` |

> 💡 *Les seuils min/max peuvent être adaptés selon le besoin (ex : -4000 à 3000 pour mieux visualiser les sommets des nuages).*

### 🏭 **Version opérationnelle**

En production, l'IR_105 est produite en TIF 16 bits pour préserver la dynamique, avec un étirement spécifique : `-scale -10000 6500 330 0`. </br>
Ainsi, nous disposons de 330 valeurs pour une amplitude de 165°C, et donc un pas de 0.5°C
| Kelvin | ° Celcius | Dynamique (`-scale`) |
|--------|------------|----------------------|
| **173,15 K** | **(-100°C)** | `-> 330` |
| **338,15 K** | **(65°C)** | `-> 0` |

In [ ]:
!gdal_translate -scale 0 330 0 255 -outsize 700 700 /stockage/DATA/TIF/20260422/globeMir105GTP.mtg.20260422.1200.24.tif.LT -ot Byte $output/IR105_mtg_20260422_1200_oper_dyn.jpg 2>/dev/null
im8 = Image.open(f"{output}/IR105_mtg_20260422_1200_oper_dyn.jpg", 'r')
display(im8)

In [ ]:
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 /stockage/DATA/TIF/20260422/globeMir105GTP.mtg.20260422.1200.24.tif.LT $output/IR105_mtg_20260422_1200_oper_france.tif #2>/dev/null
!gdal_translate -scale 0 330 0 255 -ot byte $output/IR105_mtg_20260422_1200_oper_france.tif $output/IR105_mtg_20260422_1200_oper_france.jpg
!gdal_translate -scale -10000 6500 255 0 -ot byte NETCDF:"$Mmultic1km":IR_105 $output/IR105_mtg_20260422_1200_FD_500m.tif 2>/dev/null
!gdalwarp -overwrite -t_srs '+proj=ortho lat_0=46.5 lon_0=1' -te -1120000 -630000 1120000 630000 -ts 960 540 $output/IR105_mtg_20260422_1200_FD_500m.tif $output/IR105_mtg_20260422_1200_FD_500m_france.jpg 2>/dev/null

In [ ]:
im9 = Image.open(f"{output}/IR105_mtg_20260422_1200_oper_france.jpg", 'r')
print("🏭 Version opérationnelle")
display(im9)
im10 = Image.open(f"{output}/VIS006_mtg_20260422_1200_FD_500m_france.jpg", 'r')
print("🔧 Version non corrigée")
display(im10)

Comparaison avec Synopsis

In [ ]:
im_synopsis_ir= Image.open(f"./DOCS/IR105_mtg_20260422_1200_oper_france_synopsis.jpg", 'r')
display(im_synopsis_ir)

💡**Synopsis** applique une palette de couleur de **255 teintes de gri**s pour des valeurs comprises entre -90°C et +60°C (Pas de 0.5°C donc **300 valeurs**). </br>
Dans l'image, certaines valeurs partagent la **même teinte de gris**, mais la **jauge** récupère l'information parmi les 300 valeurs du TIF.

<div id="4-extraction-de-valeurs-ponctuelles"></div>
<div class="alert alert-info alert-success">
<h3 id="4"> 4️⃣ - 📍 Extraction de valeurs ponctuelles à partir des NetCDF</h3>
</div>

### 🎯 **Commande `gdallocationinfo`**

GDAL permet d'extraire la valeur d'un pixel à des coordonnées données.

```bash
gdallocationinfo -wgs84 NETCDF:"fichier":canal longitude latitude
```

> ⚠️ **Ordre des coordonnées** : **Longitude puis latitude** (X,Y)

### 🌞 **Extraction VIS006 (réflectance)**

Pour le canal visible, les valeurs sont stockées dans les Multics en **entiers 16 bits** (Mais seulement 8 bits dans les TIF opérationnelles).

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic500m":VIS006 -3.47 48.75  2>/dev/null

💡 Les valeurs stockées dans les NetCDF (Multics ici) sont presque toujours des <b>valeurs brutes</b> auxquelles il faut appliquer une formule.
   
Le but -> gagner de la place et donc du volume </br>
Un NetCDF Multic, codé en 16 bits, pèse environ 500 Mo.
En 32 bits (Float), ce serait le double soit 1 Go.</br>
En 8 bits, ce serait seulement 250 Mo, mais on perdrait de l'information. </br>
Stockage 16 bits -> valeurs possibles : entier parmi 65 535 valeurs</br>

Pour les canaux visibles, il est indiqué dans le NetCDF:</br>
standard_name=toa_bidirectional_reflectance</br> 
scale_factor=0.01</br> 
add_offset=0</br> 

La valeur brute est ici de 6218. </br> 
Pour retrouver sa correspondance, il faut appliquer la formule </br> 
valeur = valeur brute * scale_factor + add_offset</br> 
valeur = 6218 * 0,01 + 0 = 62,18.</br> 
La reflectance est ici de 62,18 %

Cette valeur est donnée par GDAL : "Descaled Value"

**Calcul de la réflectance** :
- `Value` = 6218 (valeur brute)
- `scale_factor` = 0.01, `add_offset` = 0
- **Réflectance** = 6218 × 0.01 = **62,18 %**

> 💡 *La `Descaled Value` est directement cette valeur physique.*

### 🌡️ **Extraction IR_105 (température de brillance)**

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic1km":IR_105 -3.47 48.75  2>/dev/null

**Calcul de la température** :
- `Value` = -3655 (valeur brute)
- `scale_factor` = 0.01, `add_offset` = 273.15
- **Température (K)** = -3655 × 0.01 + 273.15 = **236,6 K**
- **Température (°C)** = -3655 x 0.01 = **-36,55 °C**

### 🔄 **Interpolation (`-r cubic`)**

Par défaut, `gdallocationinfo` renvoie la valeur du **pixel le plus proche**. Pour obtenir une valeur interpolée, utilisez `-r` :

In [ ]:
!gdallocationinfo -wgs84 -r cubic NETCDF:"$Mmultic1km":IR_105 -3.47 48.75  2>/dev/null

### ⏱️ **Extraction du temps de scan (`dtime`)**

La variable `dtime` indique le temps de scan relatif pour chaque pixel.

In [ ]:
!gdallocationinfo -wgs84 -r cubic NETCDF:"$Mmultic1km":dtime -3.47 48.75  2>/dev/null

In [ ]:
!gdalinfo NETCDF:"$Mmultic1km":dtime 2>/dev/null | grep Unit

**Interprétation** :
- Valeur brute = 5400 → 5400 × 0.1 = **540 secondes** = **9 minutes**

L'heure du slot est **12:00 UTC**. Le point a donc été scanné à **12:09 UTC**.

### 🗺️ **Visualisation des chunks de scan**

<div align="center">
<a href="../DOCS/chunks_latitude_et_heure_v20260505.jpg" target="_blank">
    <img src='../DOCS/chunks_latitude_et_heure_v20260505.jpg' alt='Scan chunks' width='800px'>
</a>
</div>

<div id="5-extraction-de-valeurs-ponctuelles"></div>
<div class="alert alert-info alert-success">
<h3 id="5"> 5 - 📍 Extraction de valeurs ponctuelles à partir des TIF</h3>
</div>

In [ ]:
IR_105="/stockage/DATA/TIF/20260422/globeMir105GTP.mtg.20260422.1200.24.tif.LT"
VIS_006="/stockage/DATA/TIF/20260422/globeMvis006GTP.mtg.20260422.1200.24.tif.LT"


print("📁 Fichiers configurés :")
print(f"   - IR : {IR_105}")
print(f"   - Visible  : {VIS_006}")

### 🌞 **Extraction VIS006 (réflectance)**


In [ ]:
!gdallocationinfo -wgs84 {VIS_006} -3.47 48.75

In [ ]:
!gdallocationinfo {VIS_006} 10655 2185  2>/dev/null

La valeur trouvée dans le TIF est de **161**. Le TIF étant en 8 bits, cette valeur est une teinte de gris parmi **255 valeurs**. </br>
Pour retrouver la reflectance correspondante, il suffit de faire un produit en croix : 161x100/255= **63,14 %**

> 💡 *Rappel : valeur trouvée dans le NetCDF :*

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic500m":VIS006 -3.47 48.75  2>/dev/null

### Valeur dans Synopsis

In [ ]:
im11 = Image.open(f"./DOCS/VIS006_mtg_20260422_1200_oper_france_synopsis_jauge.jpg", 'r')
display(im11)

### 🌡️ **Extraction IR_105 (température de brillance)** 

In [ ]:
La construction des images TIF IR est opéré en 10 bits, afin de pouvoir garder l'information avec une résolution de 0.5 °C

In [ ]:
!gdallocationinfo -wgs84 {IR_105} -3.47 48.75

In [ ]:
Temperature= 65 + (212*(-100-65)/330)
print(f" 📁 La température retrouvée est de :  {Temperature} °C")

> 💡 *Rappel : valeur trouvée dans le NetCDF :*

In [ ]:
!gdallocationinfo -wgs84 NETCDF:"$Mmultic1km":IR_105 -3.47 48.75  2>/dev/null

> *Les valeurs issues du NetCDF et du TIF sont différentes.* </br>
> ⚠️ Les TIFS produits comportent une correction de la parralaxe ! </br>
> **Il est toujours préférable de récupérer les valeurs directement dans le fichier NetCDF**

### Valeur dans Synopsis

In [ ]:
im12 = Image.open(f"./DOCS/IR105_mtg_20260422_1200_oper_france_synopsis_bis_jauge.jpg", 'r')
display(im12)

In [ ]:
im13 = Image.open(f"./DOCS/IR105_mtg_20260422_1200_oper_france_synopsis_jauge.jpg", 'r')
display(im13)

Remarque :
L'utilisation des valeurs x / y des coordonnées permets de récupérer facilement les valeurs des points autours :

In [ ]:
!gdallocationinfo {VIS_006} 10655 2185 |grep -v Band |grep -v Report  2>/dev/null # x, y
!gdallocationinfo {VIS_006} 10654 2185 |grep -v Band |grep -v Report  2>/dev/null # x-1, y
!gdallocationinfo {VIS_006} 10656 2185 |grep -v Band |grep -v Report  2>/dev/null # x+1, y
!gdallocationinfo {VIS_006} 10655 2186 |grep -v Band |grep -v Report  2>/dev/null # x, y+1
!gdallocationinfo {VIS_006} 10654 2186 |grep -v Band |grep -v Report  2>/dev/null # x-1, y+1
!gdallocationinfo {VIS_006} 10656 2186 |grep -v Band |grep -v Report  2>/dev/null # x+1, y+1
!gdallocationinfo {VIS_006} 10655 2184 |grep -v Band |grep -v Report 2>/dev/null # x, y-1
!gdallocationinfo {VIS_006} 10654 2184 |grep -v Band |grep -v Report  2>/dev/null # x-1, y-1
!gdallocationinfo {VIS_006} 10656 2184 |grep -v Band |grep -v Report  2>/dev/null # x+1, y-1


<div id="6-exemples"></div>
<div class="alert alert-info alert-success">
<h3 id="6"> 6 - 📍 Exemples réalisés à partir d'extractions</h3>
</div>

Détection et suivi de l'incendie du massif des Corbières de août 2025 </br>
Réalisation **A. Mauss** DirOP/CMS/VDS

<div align="center">
<a href="../DOCS/graph_aude_ribaute_seule.png" target="_blank">
    <img src='../DOCS/graph_aude_ribaute_seule.png' alt='Graph ribaute' width='800px'>
</a>
</div>

<div align="center">
<a href="../DOCS/graph_par_communes_incendie_aude_v2.png" target="_blank">
    <img src='../DOCS/graph_par_communes_incendie_aude_v2.png' alt='Graph ribaute' width='800px'>
</a>
</div>

<div align="center">
<a href="../DOCS/20250805_1740_test_fire_detection_ribaute.jpg" target="_blank">
    <img src='../DOCS/20250805_1740_test_fire_detection_ribaute.jpg' alt='Graph ribaute' width='600px'>
</a>
</div>

---

## ✅ **Résumé des bonnes pratiques**

| Action | Commande / Option |
|--------|-------------------|
| Visualiser un NetCDF | `gdalinfo fichier.nc` |
| Extraire un canal | `NETCDF:\"fichier.nc\":canal` |
| Appliquer un étirement | `-scale min_in max_in min_out max_out` |
| Appliquer un gamma | `-exponent 1/gamma` |
| Extraire une valeur ponctuelle | `gdallocationinfo -wgs84 ... lon lat` |
| Interpolation cubique | `-r cubic` |

---

## 🔗 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Documentation GDAL](https://gdal.org) | Référence complète |
| [Galerie CMS](http://e-sat.cms.meteo.fr/GALERIE/index.html) | Galerie CMS |
| [Comparateur d'images](http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html) | Outil de comparaison |

---

✅ **Fin du TP**